# Summary

### Bust Bands Identified

- Lower Bust Band: $28.50  

- Upper Bust Band: $31.50  

Trades were classified using the following logic:
```
df_trades['status'] = 'busted'
df_trades.loc[(df_trades['price'] > lower_bb_uber) & (df_trades['price'] < upper_bb_uber), 'status'] = 'survived'
```
### Trades Busted vs. Survived

From the provided tick data, only one trade survived the bust band filter:

- Price: $28.51  
- Quantity: 4,575 shares  
- Status: survived  

All other trades were below the lower bust band of $28.50 and were therefore classified as busted.

### Chaos Arbitrage and Profit/Loss

- Entry price of surviving trade: $28.51  

- Uber closing price on January 24, 2023: $29.93  

- Profit per share: $1.42  

Total profit calculation:

4,575 shares × $1.42 = $6,496.50

### Conclusion

Chaos arbitrage appears to have occurred. A trade executed at a price just above the lower bust band was able to remain active and, when held until the market close, generated a substantial profit. This outcome supports the hypothesis that the market anomaly on January 24, 2023 created an exploitable opportunity.


# NYSE Glitch Exercise: A Python/Pandas Investigation

- See article: https://www.reuters.com/markets/us/some-nyse-listed-stocks-briefly-halted-trading-after-market-open-2023-01-24/
- Exchange spreadsheet: 	https://www.nyse.com/publicdocs/nyse/notifications/market-status/110000531402/Final_NYSE_Auctions.xlsx
- Yahoo Finance: https://finance.yahoo.com/quote/UBER/history/

---

## The Morning Everything Went Wrong

**January 24, 2023, 9:30 AM EST**

The opening bell rang at the New York Stock Exchange, but something was terribly wrong. Over 250 stocks—including blue chips like ExxonMobil, McDonald's, and Uber—failed to execute their opening auctions. Prices went haywire. Trades that should never have happened... happened.

By the end of the day, the NYSE would bust hundreds of trades, the SEC would launch an investigation, and somewhere, in a nondescript office building, a small team of quantitative traders would be quietly celebrating.

---

## The Vultures Who Wait

Here's what most people don't know about market microstructure: **glitches are predictable in their unpredictability.**

Every major exchange has roughly one significant technical failure per year. It's not a matter of *if*, but *when*. And a small subset of hedge funds have built entire strategies around this simple fact.

These aren't your typical high-frequency traders racing to shave microseconds. These are **"chaos arbitrageurs"**—patient predators who maintain dormant systems year-round, constantly calculating one thing: *the bust bands*.

### The Bust Band Strategy

When an exchange experiences a technical failure, it doesn't simply let erroneous trades stand. Instead, it defines **bust bands**—price ranges outside of which trades will be declared null and void. The key insight? These bands follow predictable rules based on previous closing prices, reference prices, and exchange-specific rulesets.

A sophisticated fund can calculate these bands in real-time. When chaos strikes, they know exactly how far prices can deviate before trades get cancelled. They position themselves at the **edge of the abyss**—the maximum allowable price deviation—and wait.

---

## Your Mission

You are a forensic market analyst investigating whether chaos arbitrage occurred in UBER on January 24, 2023.

### Data Sources

| Source | Description |
|--------|-------------|
| NYSE Official Bust List | `https://www.nyse.com/publicdocs/nyse/notifications/market-status/110000531402/Final_NYSE_Auctions.xlsx` |
| Historical Prices | Yahoo Finance via `yfinance` |
| Tick Data | Trade fills provided below |

### Tick Data from That Morning

```python
fillstring = '''
TU|UBER:*|1465|20230124 09:30:00.174729189|25.67|12|N|h50|PREOPEN|236968|C|h08|32582
TU|UBER:*|1466|20230124 09:30:00.174734304|25.71|6206|N|h77|REGULAR|243174|C|h0A|127
TU|UBER:*|1467|20230124 09:30:00.174739474|25.79|55|N|h50|REGULAR|243229|C|h0A|32582
TU|UBER:*|1468|20230124 09:30:00.174744669|25.84|1545|N|h72|REGULAR|244774|C|h05|127
TU|UBER:*|1469|20230124 09:30:00.174750169|25.87|422|N|h72|REGULAR|245196|C|h05|127
TU|UBER:*|1470|20230124 09:30:00.174750169|28.51|4575|N|h72|REGULAR|249771|C|h05|127
'''
```

Column 4 is the fill price. Column 5 is the number of shares.

---

## Deliverables

**Using pandas, answer the following:**

1. **What were the bust bands for UBER?** Extract `Trades_Busted_Below_This_Price` and `Trades_Busted_Above_This_Price` from the NYSE spreadsheet. Note: trades *at* the boundary price are busted, so valid trades must be strictly inside.

2. **Identify which trades from the tick data were busted vs. survived.** Show your filtering logic.

3. **Did chaos arbitrage occur?** Examine the surviving trade(s) closest to the bust boundary. What is the dollar profit/loss if held until that day's close? Use `yfinance` to get UBER's closing price.


## Bust Bands


In [ ]:
import pandas as pd

excel_url = "https://www.nyse.com/publicdocs/nyse/notifications/market-status/110000531402/Final_NYSE_Auctions.xlsx"

df_nyse = pd.read_excel(excel_url)

df_nyse.head()

,SYMB,LULD_Update_Received_Time,Trades_Busted_Below_This_Price,Trades_Busted_Above_This_Price
0,ABC,2023-01-24 09:31:00.010675456,153.63400,169.80600
1,ABT,2023-01-24 09:30:00.105743104,108.11950,119.50050
2,ADM,2023-01-24 09:30:00.108570368,79.80000,88.20000
3,AI,2023-01-24 09:30:00.189420544,13.00491,15.89489
4,ALB,2023-01-24 09:30:00.124898816,244.15000,269.85000


In [ ]:
unique_symbols = df_nyse['SYMB'].unique()
print(unique_symbols)

['ABC' 'ABT' 'ADM' 'AI' 'ALB' 'ALK' 'AMC' 'AME' 'ANET' 'APTV' 'ATUS' 'AXP'
 'BA' 'BABA' 'BAX' 'BLK' 'BWXT' 'C' 'CAH' 'CAT' 'CB' 'CCI' 'CE' 'CHPT'
 'CIEN' 'CL' 'CLF' 'CRM' 'CTRA' 'CVNA' 'CVS' 'CVX' 'DAL' 'DIS' 'DLR' 'DMA'
 'DOV' 'ECL' 'EFX' 'EL' 'EMR' 'EQT' 'FCX' 'FDX' 'GAB' 'GB' 'GD' 'GE' 'GIS'
 'GM' 'GOLD' 'HAL' 'HD' 'HPE' 'IFF' 'IONQ' 'IRM' 'IVZ' 'JNJ' 'JPM' 'K'
 'KMI' 'KO' 'KSS' 'LDOS' 'LLY' 'LUMN' 'LUV' 'M' 'MA' 'MAA' 'MAIN' 'MCD'
 'MCK' 'MET' 'MGM' 'MKC' 'MMC' 'MMM' 'MMP' 'MO' 'MOS' 'MP' 'MPC' 'MPLX'
 'MPW' 'MRK' 'MRO' 'MS' 'MSI' 'MUC' 'NBXG' 'NCLH' 'NEE' 'NEM' 'NET' 'NI'
 'NINE' 'NIO' 'NKE' 'NLY' 'NNN' 'NOC' 'NOTE' 'NOW' 'NSC' 'NTR' 'NU' 'NUE'
 'NVO' 'NVTA' 'NYCB' 'NYT' 'NZF' 'OGN' 'OHI' 'OKE' 'ORCL' 'OSK' 'OXY'
 'PAXS' 'PBF' 'PBR' 'PFLT' 'PH' 'PINS' 'PLD' 'PLTR' 'PM' 'PNC' 'PPG' 'PPL'
 'PR' 'PRT' 'PRU' 'PSA' 'PSX' 'PWR' 'QBTS' 'QS' 'RA' 'RBLX' 'RIG' 'RIO'
 'ROK' 'RQI' 'RSG' 'RTX' 'RWT' 'SAND' 'SBSW' 'SCCO' 'SE' 'SHEL' 'SHOP'
 'SHW' 'SI' 'SKE' 'SLB' 'SLG' 'SMG' 'SMR' 'SNAP' 'SO' 

In [ ]:
'UBER' in unique_symbols

True

In [ ]:
bb_uber = df_nyse[df_nyse['SYMB'] == 'UBER']
bb_uber

,SYMB,LULD_Update_Received_Time,Trades_Busted_Below_This_Price,Trades_Busted_Above_This_Price
208,UBER,2023-01-24 09:30:00.175080192,28.5,31.5


In [ ]:
lower_bb_uber = bb_uber['Trades_Busted_Below_This_Price'].values[0]
upper_bb_uber = bb_uber['Trades_Busted_Above_This_Price'].values[0]
print(f'Lower BB: {lower_bb_uber}')
print(f'Upper BB: {upper_bb_uber}')

Lower BB: 28.5
Upper BB: 31.5


## Busted vs Survived

"Strictly inside a range" excluding the boundary numbers themselves.

**So the condition for a surviving trade is:**
```python
(lower_bb_uber < price) & (price < upper_bb_uber)
```

In [ ]:
import yfinance as yf
uber = yf.download(['UBER'],period='10y')

/tmp/ipython-input-3097616914.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  uber = yf.download(['UBER'],period='10y')
[*********************100%***********************]  1 of 1 completed


In [ ]:
uber.head()

Price,Close,High,Low,Open,Volume
Ticker,UBER,UBER,UBER,UBER,UBER
Date,,,,,
2019-05-10,41.570000,45.000000,41.060001,42.000000,186322500
2019-05-13,37.099998,39.240002,36.080002,38.790001,79442400
2019-05-14,39.959999,39.959999,36.849998,38.310001,46661100
2019-05-15,41.290001,41.880001,38.950001,39.369999,36086100
2019-05-16,43.000000,44.060001,41.250000,41.480000,38115500


In [ ]:
uber.columns

MultiIndex([( 'Close', 'UBER'),
            (  'High', 'UBER'),
            (   'Low', 'UBER'),
            (  'Open', 'UBER'),
            ('Volume', 'UBER')],
           names=['Price', 'Ticker'])

In [ ]:
fillstring = '''
TU|UBER:*|1465|20230124 09:30:00.174729189|25.67|12|N|h50|PREOPEN|236968|C|h08|32582
TU|UBER:*|1466|20230124 09:30:00.174734304|25.71|6206|N|h77|REGULAR|243174|C|h0A|127
TU|UBER:*|1467|20230124 09:30:00.174739474|25.79|55|N|h50|REGULAR|243229|C|h0A|32582
TU|UBER:*|1468|20230124 09:30:00.174744669|25.84|1545|N|h72|REGULAR|244774|C|h05|127
TU|UBER:*|1469|20230124 09:30:00.174750169|25.87|422|N|h72|REGULAR|245196|C|h05|127
TU|UBER:*|1470|20230124 09:30:00.174750169|28.51|4575|N|h72|REGULAR|249771|C|h05|127
'''

trade_data = []
for line in fillstring.strip().split('\n'):
    parts = line.split('|')
    timestamp = parts[3]
    price = parts[4]
    quantity = parts[5]
    trade_data.append([timestamp, price, quantity])

df_trades = pd.DataFrame(trade_data, columns=['timestamp', 'price', 'quantity'])

df_trades['timestamp'] = pd.to_datetime(df_trades['timestamp'])

df_trades['price'] = df_trades['price'].astype(float)
df_trades['quantity'] = df_trades['quantity'].astype(int)

print(df_trades)

                      timestamp  price  quantity
0 2023-01-24 09:30:00.174729189  25.67        12
1 2023-01-24 09:30:00.174734304  25.71      6206
2 2023-01-24 09:30:00.174739474  25.79        55
3 2023-01-24 09:30:00.174744669  25.84      1545
4 2023-01-24 09:30:00.174750169  25.87       422
5 2023-01-24 09:30:00.174750169  28.51      4575


In [ ]:
df_trades['status'] = 'busted'
df_trades.loc[(df_trades['price'] > lower_bb_uber) & (df_trades['price'] < upper_bb_uber), 'status'] = 'survived'

print("Trades with bust status:")
print(df_trades)

Trades with bust status:
                      timestamp  price  quantity    status
0 2023-01-24 09:30:00.174729189  25.67        12    busted
1 2023-01-24 09:30:00.174734304  25.71      6206    busted
2 2023-01-24 09:30:00.174739474  25.79        55    busted
3 2023-01-24 09:30:00.174744669  25.84      1545    busted
4 2023-01-24 09:30:00.174750169  25.87       422    busted
5 2023-01-24 09:30:00.174750169  28.51      4575  survived


## Choas Arbitrage

In [ ]:
survived_trades = df_trades[df_trades['status'] == 'survived'].copy()
print("Survived Trades:")
print(survived_trades)

Survived Trades:
                      timestamp  price  quantity    status
5 2023-01-24 09:30:00.174750169  28.51      4575  survived


In [ ]:
if not survived_trades.empty:
    # Since there's only one surviving trade in this example, it's both closest to lower and upper
    closest_to_lower_price = survived_trades.iloc[0]['price']
    closest_to_lower_quantity = survived_trades.iloc[0]['quantity']

    closest_to_upper_price = survived_trades.iloc[0]['price']
    closest_to_upper_quantity = survived_trades.iloc[0]['quantity']

    print(f"Surviving trade closest to lower bust band: Price={closest_to_lower_price}, Quantity={closest_to_lower_quantity}")
    print(f"Surviving trade closest to upper bust band: Price={closest_to_upper_price}, Quantity={closest_to_upper_quantity}")
else:
    print("No trades survived.")

date_to_retrieve = survived_trades.iloc[0]["timestamp"]
uber_closing_price = uber['Close', 'UBER'].loc[date_to_retrieve.normalize()]
print(f"UBER's closing price on {date_to_retrieve}: {uber_closing_price}")

Surviving trade closest to lower bust band: Price=28.51, Quantity=4575
Surviving trade closest to upper bust band: Price=28.51, Quantity=4575
UBER's closing price on 2023-01-24 09:30:00.174750169: 29.93000030517578


In [ ]:
if not survived_trades.empty:
    profit_loss_per_share = uber_closing_price - closest_to_lower_price
    total_profit_loss = profit_loss_per_share * closest_to_lower_quantity

    print(f"\nProfit/Loss Calculation for the surviving trade (Price={closest_to_lower_price}, Quantity={closest_to_lower_quantity}):")
    print(f"  Closing Price on 2023-01-24: {uber_closing_price:.2f}")
    print(f"  Trade Price: {closest_to_lower_price:.2f}")
    print(f"  Profit/Loss per share: {profit_loss_per_share:.2f}")
    print(f"  Total Profit/Loss: {total_profit_loss:.2f}")
else:
    print("\nCannot calculate profit/loss as no trades survived.")


Profit/Loss Calculation for the surviving trade (Price=28.51, Quantity=4575):
  Closing Price on 2023-01-24: 29.93
  Trade Price: 28.51
  Profit/Loss per share: 1.42
  Total Profit/Loss: 6496.50
